In [1]:
# ============================================================
# CELL 0 — Install Required Packages
# ============================================================

%pip install -q torch-geometric
%pip install torch

Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 3.4 MB/s eta 0:00:37
   ---------------------------------------- 1.0/124.1 MB 2.4 MB/s eta 0:00:52
    --------------------------------------- 1.8/124.1 MB 2.6 MB/s eta 0:00:48
    --------------------------------------- 2.4/124.1 MB 2.9 MB/s eta 0:00:42
   - -------------------------------------- 3.1/124.1 MB 2.8 MB/s eta 0:00:44
   - -------------------------------------- 3.7/124.1 MB 2.8 MB/s eta 0:00:44
   - -------------------------------------- 4.5/124.1 MB 2.9 MB/s eta 0:00:41
   - -------------------------------------- 5.2/124.1 MB 3.0 MB/s eta 0:00:40
   -- ------------------------------------- 6.8/124.1 MB 3.6 MB/s eta 0:00:33
   -- ------

In [2]:
# ============================================================
# CELL 1 — Setup
# ============================================================

from pathlib import Path
import random, joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GATv2Conv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LEN, BATCH_SIZE, EPOCHS = 24, 128, 35
LR, WEIGHT_DECAY = 1e-3, 1e-3

TARGETS = ["co","no2","o3","pm10","pm25","so2"]
FEATURES = [
    "latitude","longitude","era5_t2m_c","era5_d2m_c","era5_sp_hpa","era5_tp_mm","era5_wind_speed","era5_wind_dir","era5_blh",
    "hour","month","dayofweek","hour_sin","hour_cos","dow_sin","dow_cos",
    "co_lag1","co_lag24","co_roll24_mean","no2_lag1","no2_lag24","no2_roll24_mean",
    "o3_lag1","o3_lag24","o3_roll24_mean","pm10_lag1","pm10_lag24","pm10_roll24_mean",
    "pm25_lag1","pm25_lag24","pm25_roll24_mean","so2_lag1","so2_lag24","so2_roll24_mean"
]

print(f"Device: {DEVICE}")
print(f"Features: {len(FEATURES)} | Targets: {len(TARGETS)}")

C:\Users\vabhi\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\vabhi\AppData\Roaming\Python\Python311\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device: cpu
Features: 34 | Targets: 6


In [3]:
# ============================================================
# CELL 2 — Load Data + Graph
# ============================================================

import networkx as nx

DATA_DIR = ROOT / "data/processed"
GRAPH_DIR = ROOT / "graph"

train = pd.read_csv(DATA_DIR/"train.csv", parse_dates=["datetime_utc"])
val = pd.read_csv(DATA_DIR/"val.csv", parse_dates=["datetime_utc"])
test = pd.read_csv(DATA_DIR/"test.csv", parse_dates=["datetime_utc"])

station_df = pd.read_csv(GRAPH_DIR/"station_metadata_final.csv")
edge_df = pd.read_csv(GRAPH_DIR/"edge_index_final.csv")

STATIONS = station_df.location_id.tolist()
N = len(STATIONS)
loc2idx = {s:i for i,s in enumerate(STATIONS)}

BASE_EDGE_INDEX = np.stack([
    edge_df.source.map(loc2idx).to_numpy(),
    edge_df.target.map(loc2idx).to_numpy()
])

EDGE_INDEX = torch.tensor(BASE_EDGE_INDEX, dtype=torch.long, device=DEVICE)
EDGE_ATTR = torch.tensor(edge_df.distance_km.to_numpy(), dtype=torch.float32, device=DEVICE).view(-1,1)

G = nx.Graph()
G.add_nodes_from(STATIONS)
G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))

print("="*60)
print("GRAPH + DATA LOADED")
print("="*60)
print(f"Stations: {N}")
print(f"Directed edges: {EDGE_INDEX.shape[1]}")
print(f"Connected components: {nx.number_connected_components(G)}")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

GRAPH + DATA LOADED
Stations: 37
Directed edges: 169
Connected components: 1
Train: 1,134,975 | Val: 243,201 | Test: 243,238


In [4]:
# ============================================================
# CELL 3 — Build Grids + Scaling
# ============================================================

def build_grid(df):
    times = pd.date_range(df.datetime_utc.min(), df.datetime_utc.max(), freq="h")
    tm = {t:i for i,t in enumerate(times)}
    sm = {s:i for i,s in enumerate(STATIONS)}
    X = np.full((len(times),N,len(FEATURES)), np.nan, np.float32)
    Y = np.full((len(times),N,6), np.nan, np.float32)
    M = np.zeros_like(Y)

    d = df[df.location_id.isin(STATIONS)]
    ti, si = d.datetime_utc.map(tm).to_numpy(), d.location_id.map(sm).to_numpy()
    X[ti,si] = d[FEATURES].to_numpy(np.float32)

    for k,t in enumerate(TARGETS):
        v = d[t].to_numpy(np.float32); ok = np.isfinite(v)
        Y[ti[ok],si[ok],k], M[ti[ok],si[ok],k] = v[ok], 1

    return times, X, Y, M

train_time,train_X,train_Y,train_M = build_grid(train)
val_time,val_X,val_Y,val_M = build_grid(val)
test_time,test_X,test_Y,test_M = build_grid(test)

fill_values = np.nan_to_num(np.nanmedian(train_X.reshape(-1,len(FEATURES)),axis=0),nan=0.0)

def fill(X):
    z = X.reshape(-1,len(FEATURES)).copy()
    r,c = np.where(~np.isfinite(z)); z[r,c] = fill_values[c]
    return z.reshape(X.shape).astype(np.float32)

train_X,val_X,test_X = map(fill,[train_X,val_X,test_X])

feature_scaler = StandardScaler()
train_X = feature_scaler.fit_transform(train_X.reshape(-1,len(FEATURES))).reshape(train_X.shape).astype(np.float32)
val_X = feature_scaler.transform(val_X.reshape(-1,len(FEATURES))).reshape(val_X.shape).astype(np.float32)
test_X = feature_scaler.transform(test_X.reshape(-1,len(FEATURES))).reshape(test_X.shape).astype(np.float32)

target_mean = np.nanmean(train_Y.reshape(-1,6),axis=0)
target_std = np.where(np.nanstd(train_Y.reshape(-1,6),axis=0)>0,np.nanstd(train_Y.reshape(-1,6),axis=0),1)

def scale_y(Y): return ((Y-target_mean)/target_std).astype(np.float32)
def inv_y(Y): return Y*target_std + target_mean

train_Y,val_Y,test_Y = map(scale_y,[train_Y,val_Y,test_Y])

print("Training-only scaling complete.")
print(f"Train: {train_X.shape}")
print(f"Val  : {val_X.shape}")
print(f"Test : {test_X.shape}")

Training-only scaling complete.
Train: (30675, 37, 34)
Val  : (6573, 37, 34)
Test : (6574, 37, 34)


In [5]:
# ============================================================
# CELL 4 — Pollutant History + Residual Targets
# ============================================================

def augment(X,Y,M): return np.concatenate([X,np.nan_to_num(Y,nan=0),M],-1).astype(np.float32)

train_Xa,val_Xa,test_Xa = [augment(X,Y,M) for X,Y,M in [
    (train_X,train_Y,train_M),(val_X,val_Y,val_M),(test_X,test_Y,test_M)
]]

raw_train,raw_val,raw_test = [inv_y(Y) for Y in [train_Y,val_Y,test_Y]]

def make_res(Y,M):
    R = np.full_like(Y,np.nan); R[1:] = Y[1:] - Y[:-1]
    RM = np.zeros_like(M); RM[1:] = M[1:] * M[:-1]
    return R,RM

train_R,train_RM = make_res(raw_train,train_M)
val_R,val_RM = make_res(raw_val,val_M)
test_R,test_RM = make_res(raw_test,test_M)

res_mean = np.nanmean(train_R.reshape(-1,6),0)
res_std = np.where(np.nanstd(train_R.reshape(-1,6),0)>0,np.nanstd(train_R.reshape(-1,6),0),1)

def scale_r(R): return ((R-res_mean)/res_std).astype(np.float32)

train_R,val_R,test_R = map(scale_r,[train_R,val_R,test_R])
INPUT_DIM = train_Xa.shape[-1]

print(f"Final input features: {INPUT_DIM}")
print("Residual targets prepared.")

Final input features: 46
Residual targets prepared.


In [6]:
# ============================================================
# CELL 5 — Sequence Dataloaders
# ============================================================

class RDataset(Dataset):
    def __init__(self,X,R,RM,Y):
        self.X,self.R,self.RM,self.Y = X,R,RM,Y
        self.t = np.arange(SEQ_LEN,len(X))
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        t = self.t[i]
        return self.X[t-SEQ_LEN:t],self.R[t],self.RM[t],self.Y[t-1]

BASE_EDGE_CPU = BASE_EDGE_INDEX.astype(np.int64)

def collate(b):
    X,R,RM,last = map(np.stack,zip(*b))
    X,R,RM,last = [torch.tensor(x,dtype=torch.float32) for x in [X,R,RM,last]]
    B,T,Nn,Fdim = X.shape
    off = np.arange(B)*Nn
    src = BASE_EDGE_CPU[0][None,:] + off[:,None]
    dst = BASE_EDGE_CPU[1][None,:] + off[:,None]
    ei = torch.tensor(np.vstack([src.ravel(),dst.ravel()]),dtype=torch.long)
    return {"x_grid":X,"edge_index":ei,"edge_attr":EDGE_ATTR.cpu().repeat(B,1),"y":R,"mask":RM,"last":last}

train_loader = DataLoader(RDataset(train_Xa,train_R,train_RM,raw_train),BATCH_SIZE,shuffle=True,collate_fn=collate)
val_loader = DataLoader(RDataset(val_Xa,val_R,val_RM,raw_val),BATCH_SIZE,shuffle=False,collate_fn=collate)
test_loader = DataLoader(RDataset(test_Xa,test_R,test_RM,raw_test),BATCH_SIZE,shuffle=False,collate_fn=collate)

print(f"Samples: {len(train_loader.dataset):,} / {len(val_loader.dataset):,} / {len(test_loader.dataset):,}")

Samples: 30,651 / 6,549 / 6,550


In [7]:
# ============================================================
# CELL 6 — Final Residual GAT-LSTM
# ============================================================

class ResidualGATLSTM(nn.Module):
    def __init__(self,Fdim,hidden=128,heads=4,drop=0.2):
        super().__init__()
        self.lstm = nn.LSTM(Fdim,hidden,2,batch_first=True,dropout=drop)
        self.emb = nn.Embedding(N,16)
        self.gat1 = GATv2Conv(hidden+16,hidden//heads,heads=heads,edge_dim=1)
        self.gat2 = GATv2Conv(hidden,hidden,edge_dim=1)
        self.n1 = nn.LayerNorm(hidden); self.n2 = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(drop)
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden,64),nn.SiLU(),nn.Dropout(drop),nn.Linear(64,1))
            for _ in TARGETS
        ])

    def forward(self,b):
        B,T,Nn,Fdim = b["x_grid"].shape
        x = b["x_grid"].permute(0,2,1,3).reshape(B*Nn,T,Fdim)
        h,_ = self.lstm(x); h = h[:,-1].view(B,Nn,-1)
        e = self.emb(torch.arange(N,device=h.device)).expand(B,-1,-1)
        h = torch.cat([h,e],-1).reshape(B*Nn,-1)
        off = torch.arange(B,device=h.device)*Nn
        ei = torch.cat([EDGE_INDEX.to(h.device)+o for o in off],1)
        ea = EDGE_ATTR.to(h.device).repeat(B,1)
        z = F.elu(self.gat1(h,ei,edge_attr=ea))
        z = self.drop(self.n1(z))
        z = self.gat2(z,ei,edge_attr=ea)
        z = self.n2(z+h[:,:z.shape[1]])
        z = self.drop(z).view(B,Nn,-1)
        return torch.cat([head(z) for head in self.heads],-1)

model = ResidualGATLSTM(INPUT_DIM).to(DEVICE)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 344,150


In [8]:
# ============================================================
# CELL 7 — Load Existing Trained GAT-LSTM
# ============================================================

MODELS_DIR = ROOT / "models"
CHECKPOINT = MODELS_DIR / "best_improved_gat_lstm.pt"

if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Trained checkpoint not found: {CHECKPOINT}"
    )

try:
    state_dict = torch.load(
        CHECKPOINT,
        map_location=DEVICE,
        weights_only=True
    )
except TypeError:
    # Compatibility with older PyTorch versions.
    state_dict = torch.load(
        CHECKPOINT,
        map_location=DEVICE
    )

model.load_state_dict(state_dict, strict=True)
model.eval()

print("=" * 60)
print("EXISTING GAT-LSTM CHECKPOINT LOADED")
print("=" * 60)
print("Checkpoint:", CHECKPOINT)
print("Training skipped.")
print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

EXISTING GAT-LSTM CHECKPOINT LOADED
Checkpoint: c:\Users\vabhi\Downloads\Implementation\Implementation\models\best_improved_gat_lstm.pt
Training skipped.
Parameters: 344,150


In [14]:
# ============================================================
# CELL 8 — Final Evaluation
# ============================================================

AQI_CATS = [
    "Good", "Satisfactory", "Moderately polluted",
    "Poor", "Very Poor", "Severe"
]

AQI_BP = {
    "pm10": [
        (0, 50, 0, 50), (50, 100, 50, 100),
        (100, 250, 100, 200), (250, 350, 200, 300),
        (350, 430, 300, 400), (430, np.inf, 400, 500)
    ],
    "pm25": [
        (0, 30, 0, 50), (30, 60, 50, 100),
        (60, 90, 100, 200), (90, 120, 200, 300),
        (120, 250, 300, 400), (250, np.inf, 400, 500)
    ],
    "no2": [
        (0, 40, 0, 50), (40, 80, 50, 100),
        (80, 180, 100, 200), (180, 280, 200, 300),
        (280, 400, 300, 400), (400, np.inf, 400, 500)
    ],
    "o3": [
        (0, 50, 0, 50), (50, 100, 50, 100),
        (100, 168, 100, 200), (168, 208, 200, 300),
        (208, 748, 300, 400), (748, np.inf, 400, 500)
    ],
    "co": [
        (0, 1, 0, 50), (1, 2, 50, 100),
        (2, 10, 100, 200), (10, 17, 200, 300),
        (17, 34, 300, 400), (34, np.inf, 400, 500)
    ],
    "so2": [
        (0, 40, 0, 50), (40, 80, 50, 100),
        (80, 380, 100, 200), (380, 800, 200, 300),
        (800, 1600, 300, 400), (1600, np.inf, 400, 500)
    ],
}

AQI_WINDOW_HOURS = {
    "co": 8,
    "no2": 24,
    "o3": 8,
    "pm10": 24,
    "pm25": 24,
    "so2": 24,
}


def subindex(values, pollutant):
    values = np.asarray(values, dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)

    valid_values = np.isfinite(values) & (values >= 0)

    for i, (lo, hi, ilo, ihi) in enumerate(AQI_BP[pollutant]):

        if i == 0:
            in_band = (values >= lo) & (values <= hi)
        elif np.isfinite(hi):
            in_band = (values > lo) & (values <= hi)
        else:
            in_band = values > lo

        use = valid_values & in_band & np.isnan(out)

        if not np.any(use):
            continue

        if np.isinf(hi):
            out[use] = 500.0
        else:
            out[use] = (
                ilo
                + (ihi - ilo) * (values[use] - lo) / (hi - lo)
            )

    return np.floor(out)


def rolling_mean_complete(values, valid_mask, window):
    values = np.asarray(values, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    if values.ndim != 2 or valid_mask.shape != values.shape:
        raise ValueError(
            "values and valid_mask must have shape (time, stations)."
        )

    means = np.full(values.shape, np.nan, dtype=float)
    out_mask = np.zeros(values.shape, dtype=bool)

    for t in range(window - 1, len(values)):

        block = values[t - window + 1:t + 1]
        block_mask = (
            valid_mask[t - window + 1:t + 1]
            & np.isfinite(block)
        )

        station_ok = block_mask.all(axis=0)

        if station_ok.any():
            means[t, station_ok] = block[:, station_ok].mean(axis=0)
            out_mask[t, station_ok] = True

    return means, out_mask



def evaluate(loader, rawY, split):

    model.eval()
    P, M, L = [], [], []

    with torch.no_grad():
        for b in loader:
            bt = {k: v.to(DEVICE) for k, v in b.items()}

            P.append(model(bt).cpu().numpy())
            M.append(b["mask"].numpy())
            L.append(b["last"].numpy())

    P = np.concatenate(P)
    M = np.concatenate(M).astype(bool)
    L = np.nan_to_num(np.concatenate(L), nan=0.0)

    # Reconstruct predicted concentrations in their original units.
    # Keep these original predictions unchanged for regression metrics.
    P = P * res_std + res_mean + L

    Y = rawY[SEQ_LEN:]

    if not (P.shape == Y.shape == M.shape):
        raise ValueError(
            f"{split}: prediction, target and mask shapes differ: "
            f"P={P.shape}, Y={Y.shape}, M={M.shape}"
        )

    # --------------------------------------------------------
    # 1. Regression metrics
    # --------------------------------------------------------
    regression_rows = []

    for k, pollutant in enumerate(TARGETS):

        valid = (
            M[:, :, k]
            & np.isfinite(Y[:, :, k])
            & np.isfinite(P[:, :, k])
        )

        yt = Y[:, :, k][valid]
        yp = P[:, :, k][valid]

        if len(yt) == 0:
            raise ValueError(
                f"{split}: no valid regression observations for {pollutant}."
            )

        regression_rows.append({
            "Model": "Improved Residual GAT-LSTM",
            "Split": split,
            "Pollutant": pollutant,
            "MAE": mean_absolute_error(yt, yp),
            "RMSE": np.sqrt(mean_squared_error(yt, yp)),
            "R2": r2_score(yt, yp),
        })

    # --------------------------------------------------------
    # 2. CPCB-period AQI rolling averages
    # --------------------------------------------------------
    is_validation = split.lower().startswith("val")

    source_df = val if is_validation else test
    source_times = val_time if is_validation else test_time

    source_times = pd.DatetimeIndex(
        pd.to_datetime(source_times, utc=True).tz_localize(None)
    )

    q_actual = source_df[
        ["datetime_utc", "location_id"] + TARGETS
    ].copy()

    q_actual["time_key"] = (
        pd.to_datetime(q_actual["datetime_utc"], utc=True)
        .dt.tz_localize(None)
    )

    q_actual["loc_key"] = q_actual["location_id"].astype(str)

    q_actual = q_actual.sort_values(
        ["loc_key", "time_key"]
    ).reset_index(drop=True)

    if q_actual.duplicated(["loc_key", "time_key"]).any():
        raise ValueError(
            f"{split}: duplicate station-time rows in source data."
        )

    # Use exactly the same Pandas rolling calculation as Notebook 6.
    for pollutant in TARGETS:

        window = AQI_WINDOW_HOURS[pollutant]

        q_actual[f"actual_roll__{pollutant}"] = (
            q_actual.groupby(
                "loc_key",
                sort=False,
            )[pollutant]
            .transform(
                lambda s: s.rolling(
                    window=window,
                    min_periods=window,
                ).mean()
            )
        )

    # Map rolling actual concentrations into the model's
    # (time, station, pollutant) layout.
    time_to_idx = {
        pd.Timestamp(t): i
        for i, t in enumerate(source_times)
    }

    station_to_idx = {
        str(station): i
        for i, station in enumerate(STATIONS)
    }

    q_actual["_time_idx"] = q_actual["time_key"].map(time_to_idx)
    q_actual["_station_idx"] = q_actual["loc_key"].map(station_to_idx)

    if (
        q_actual["_time_idx"].isna().any()
        or q_actual["_station_idx"].isna().any()
    ):
        raise ValueError(
            f"{split}: some actual rolling values could not be "
            "mapped to the model time/station grid."
        )

    ti = q_actual["_time_idx"].to_numpy(dtype=np.int64)
    si = q_actual["_station_idx"].to_numpy(dtype=np.int64)

    actual_roll_full = np.full(
        (len(source_times), N, len(TARGETS)),
        np.nan,
        dtype=np.float64,
    )

    for k, pollutant in enumerate(TARGETS):
        actual_roll_full[ti, si, k] = q_actual[
            f"actual_roll__{pollutant}"
        ].to_numpy(dtype=float)

    # Model predictions begin after SEQ_LEN input observations.
    actual_roll = actual_roll_full[SEQ_LEN:]
    actual_roll_mask = np.isfinite(actual_roll)

    # AQI-only predictions: clip negative hourly concentrations
    # before rolling averages, consistently with Notebook 6.
    pred_for_aqi = np.clip(P, 0.0, None)
    pred_valid_mask = M & np.isfinite(P)
    pred_for_aqi[~pred_valid_mask] = np.nan

    pred_roll = np.full(P.shape, np.nan, dtype=float)
    pred_roll_mask = np.zeros(P.shape, dtype=bool)

    for k, pollutant in enumerate(TARGETS):

        window = AQI_WINDOW_HOURS[pollutant]

        pred_roll[:, :, k], pred_roll_mask[:, :, k] = (
            rolling_mean_complete(
                pred_for_aqi[:, :, k],
                pred_valid_mask[:, :, k],
                window,
            )
        )

    if actual_roll.shape != P.shape:
        raise ValueError(
            f"{split}: actual rolling shape {actual_roll.shape} "
            f"does not match predictions {P.shape}."
        )


    # --------------------------------------------------------
    # 3. Common AQI classification cohort
    # --------------------------------------------------------
    # One station-time window is retained only if all six actual
    # pollutant averages AND all six predicted averages are valid.
    aqi_valid = (
        actual_roll_mask.all(axis=2)
        & pred_roll_mask.all(axis=2)
    )

    candidate_count = int(aqi_valid.sum())

    if candidate_count == 0:
        raise ValueError(
            f"{split}: no complete CPCB-period AQI windows remain."
        )

    actual_concentrations = actual_roll[aqi_valid]
    predicted_concentrations = pred_roll[aqi_valid]

    true_si = np.column_stack([
        subindex(actual_concentrations[:, k], pollutant)
        for k, pollutant in enumerate(TARGETS)
    ])

    pred_si = np.column_stack([
        subindex(predicted_concentrations[:, k], pollutant)
        for k, pollutant in enumerate(TARGETS)
    ])

    # Do not silently change the cohort after selecting complete windows.
    # If this fails, inspect breakpoint definitions or invalid concentrations.
    invalid_subindices = (
        ~np.isfinite(true_si).all(axis=1)
        | ~np.isfinite(pred_si).all(axis=1)
    )

    if invalid_subindices.any():
        raise ValueError(
            f"{split}: {int(invalid_subindices.sum()):,} of "
            f"{candidate_count:,} complete AQI windows have invalid "
            "sub-indices. Check actual concentrations and AQI breakpoints."
        )

    true_aqi = np.max(true_si, axis=1)
    pred_aqi = np.max(pred_si, axis=1)

    bins = [-np.inf, 50, 100, 200, 300, 400, np.inf]

    tc = pd.cut(
        true_aqi,
        bins=bins,
        labels=AQI_CATS,
        include_lowest=True,
    )

    pc = pd.cut(
        pred_aqi,
        bins=bins,
        labels=AQI_CATS,
        include_lowest=True,
    )

    if pd.isna(tc).any() or pd.isna(pc).any():
        raise ValueError(
            f"{split}: AQI categories contain missing values."
        )

    # Convert both category arrays to strings for sklearn compatibility.
    y_true_cat = pd.Series(tc).astype(str).to_numpy()
    y_pred_cat = pd.Series(pc).astype(str).to_numpy()

    aqi_metrics = {
        "Accuracy": accuracy_score(y_true_cat, y_pred_cat),

        "Precision": precision_score(
            y_true_cat,
            y_pred_cat,
            labels=AQI_CATS,
            average="macro",
            zero_division=0,
        ),

        "Recall": recall_score(
            y_true_cat,
            y_pred_cat,
            labels=AQI_CATS,
            average="macro",
            zero_division=0,
        ),

        "F1": f1_score(
            y_true_cat,
            y_pred_cat,
            labels=AQI_CATS,
            average="macro",
            zero_division=0,
        ),

        "AQI_samples": len(y_true_cat),
    }

    # --------------------------------------------------------
    # 4. Save station/time identities for exact window comparison
    # --------------------------------------------------------
    # Output arrays begin at split_times[SEQ_LEN].
    output_times = pd.DatetimeIndex(source_times[SEQ_LEN:])

    if len(output_times) != Y.shape[0]:
        raise ValueError(
            f"{split}: time dimension does not match AQI arrays."
        )

    time_grid = np.repeat(
        output_times.to_numpy(dtype="datetime64[ns]"),
        N,
    )

    station_grid = np.tile(
        np.asarray(STATIONS, dtype=str),
        len(output_times),
    )

    selected = aqi_valid.ravel()

    aqi_windows = pd.DataFrame({
        "datetime_utc": time_grid[selected],
        "location_id": station_grid[selected],
        "actual_AQI": true_aqi,
        "predicted_AQI": pred_aqi,
        "y_true_aqi": y_true_cat,
        "y_pred_aqi": y_pred_cat,
    })

    print(
        f"{split}: complete common-style AQI windows = "
        f"{len(aqi_windows):,}"
    )
    print(f"{split}: invalid AQI categories = 0")

    return (
        pd.DataFrame(regression_rows),
        aqi_metrics,
        P,
        Y,
        M,
        aqi_windows,
    )


# ------------------------------------------------------------
# Evaluate the existing checkpoint on validation and test data
# ------------------------------------------------------------

(
    val_results,
    val_aqi,
    val_pred,
    val_true,
    val_mask,
    val_aqi_windows,
) = evaluate(
    val_loader,
    raw_val,
    "Validation",
)

(
    test_results,
    test_aqi,
    test_pred,
    test_true,
    test_mask,
    test_aqi_windows,
) = evaluate(
    test_loader,
    raw_test,
    "Test",
)

print("=" * 60)
print("TEST REGRESSION")
print("=" * 60)
print(test_results.to_string(index=False))

print("\nTEST CPCB-PERIOD AQI CLASSIFICATION")
print(pd.Series(test_aqi))

print("\nValidation AQI windows:", f"{len(val_aqi_windows):,}")
print("Test AQI windows:", f"{len(test_aqi_windows):,}")


Validation: complete common-style AQI windows = 88,853
Validation: invalid AQI categories = 0
Test: complete common-style AQI windows = 84,014
Test: invalid AQI categories = 0
TEST REGRESSION
                     Model Split Pollutant       MAE      RMSE       R2
Improved Residual GAT-LSTM  Test        co  0.154622  0.340344 0.875074
Improved Residual GAT-LSTM  Test       no2  4.957168  9.168872 0.925033
Improved Residual GAT-LSTM  Test        o3  5.159182 10.976510 0.919588
Improved Residual GAT-LSTM  Test      pm10 22.283785 41.372500 0.920323
Improved Residual GAT-LSTM  Test      pm25 11.566792 23.705875 0.940375
Improved Residual GAT-LSTM  Test       so2  2.240624  5.449157 0.832959

TEST CPCB-PERIOD AQI CLASSIFICATION
Accuracy           0.957912
Precision          0.957895
Recall             0.939591
F1                 0.948343
AQI_samples    84014.000000
dtype: float64

Validation AQI windows: 88,853
Test AQI windows: 84,014


In [15]:
# ============================================================
# CELL 9 — Save Final Results & Artifacts
# ============================================================

METRICS_DIR = ROOT / "metrics"
MODELS_DIR = ROOT / "models"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

reg = pd.concat(
    [val_results, test_results],
    ignore_index=True
)

macro = {
    "Model": "Improved Residual GAT-LSTM",
    "MAE": test_results.MAE.mean(),
    "RMSE": test_results.RMSE.mean(),
    "R2": test_results.R2.mean(),
    **test_aqi
}

pd.DataFrame([macro]).to_csv(
    METRICS_DIR / "gat_lstm_final_7_metrics.csv",
    index=False
)

reg.to_csv(
    METRICS_DIR / "gat_lstm_final_per_pollutant.csv",
    index=False
)


# Save the station-time windows used for AQI classification.
val_aqi_windows.to_csv(
    METRICS_DIR / "gat_lstm_validation_aqi_windows.csv",
    index=False,
)

test_aqi_windows.to_csv(
    METRICS_DIR / "gat_lstm_test_aqi_windows.csv",
    index=False,
)

print(
    "Validation AQI windows saved:",
    METRICS_DIR / "gat_lstm_validation_aqi_windows.csv",
)
print(
    "Test AQI windows saved:",
    METRICS_DIR / "gat_lstm_test_aqi_windows.csv",
)

np.save(
    MODELS_DIR / "gat_lstm_test_predictions.npy",
    test_pred
)

np.save(
    MODELS_DIR / "gat_lstm_test_targets.npy",
    test_true
)

np.save(
    MODELS_DIR / "gat_lstm_test_mask.npy",
    test_mask
)

np.save(
    MODELS_DIR / "gat_lstm_test_times.npy",
    np.asarray(
        test_time[SEQ_LEN:],
        dtype="datetime64[ns]"
    )
)

torch.save(
    model.state_dict(),
    MODELS_DIR / "best_improved_gat_lstm.pt"
)

joblib.dump(
    feature_scaler,
    MODELS_DIR / "feature_scaler.pkl"
)

joblib.dump({
    "features": FEATURES,
    "targets": TARGETS,
    "seq_len": SEQ_LEN,
    "station_list": STATIONS,
    "fill_values": fill_values,
    "target_mean": target_mean,
    "target_std": target_std,
    "res_mean": res_mean,
    "res_std": res_std,
    "aqi_breakpoints": AQI_BP,
    "aqi_window_hours": AQI_WINDOW_HOURS,
}, MODELS_DIR / "gat_lstm_config.pkl")

print("=" * 60)
print("FINAL ARTIFACTS SAVED")
print("=" * 60)

for path in [
    METRICS_DIR / "gat_lstm_final_7_metrics.csv",
    METRICS_DIR / "gat_lstm_final_per_pollutant.csv",
    MODELS_DIR / "gat_lstm_test_predictions.npy",
    MODELS_DIR / "gat_lstm_test_targets.npy",
    MODELS_DIR / "gat_lstm_test_mask.npy",
    MODELS_DIR / "gat_lstm_test_times.npy",
    MODELS_DIR / "best_improved_gat_lstm.pt",
    MODELS_DIR / "feature_scaler.pkl",
    MODELS_DIR / "gat_lstm_config.pkl",
]:
    print("✓", path)

Validation AQI windows saved: c:\Users\vabhi\Downloads\Implementation\Implementation\metrics\gat_lstm_validation_aqi_windows.csv
Test AQI windows saved: c:\Users\vabhi\Downloads\Implementation\Implementation\metrics\gat_lstm_test_aqi_windows.csv
FINAL ARTIFACTS SAVED
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\metrics\gat_lstm_final_7_metrics.csv
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\metrics\gat_lstm_final_per_pollutant.csv
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\models\gat_lstm_test_predictions.npy
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\models\gat_lstm_test_targets.npy
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\models\gat_lstm_test_mask.npy
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\models\gat_lstm_test_times.npy
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\models\best_improved_gat_lstm.pt
✓ c:\Users\vabhi\Downloads\Implementation\Implementation\models\feature_scaler.pkl
✓ c:\Use

In [16]:

# ============================================================
# CELL 10 — Verify Notebook 7 vs Notebook 6 AQI windows
# ============================================================

reference_path = (
    ROOT
    / "metrics"
    / "baseline_fulltrain"
    / "common_aqi_windows_all_models.csv"
)

current_path = METRICS_DIR / "gat_lstm_test_aqi_windows.csv"

if not reference_path.exists():
    raise FileNotFoundError(
        "Notebook 6 AQI comparison file not found:\n"
        f"{reference_path}"
    )

reference = pd.read_csv(reference_path)
current = pd.read_csv(current_path)

key_cols = ["datetime_utc", "location_id"]

for frame in [reference, current]:
    frame["datetime_utc"] = (
        pd.to_datetime(frame["datetime_utc"], utc=True)
        .dt.tz_localize(None)
    )
    frame["location_id"] = frame["location_id"].astype(str)

if reference.duplicated(key_cols).any():
    raise ValueError("Notebook 6 contains duplicate station-time keys.")

if current.duplicated(key_cols).any():
    raise ValueError("Notebook 7 contains duplicate station-time keys.")

comparison = reference.merge(
    current,
    on=key_cols,
    how="outer",
    indicator=True,
    suffixes=("_nb6", "_nb7"),
    validate="one_to_one",
)

same_keys = (
    len(reference) == len(current)
    and comparison["_merge"].eq("both").all()
)

print("Notebook 6 windows:", f"{len(reference):,}")
print("Notebook 7 windows:", f"{len(current):,}")
print("Identical station-time window sets:", same_keys)

if not same_keys:
    print("\nWindow membership differences:")
    print(comparison["_merge"].value_counts(dropna=False))
else:
    checks = {
        "Actual AQI values match": np.allclose(
            comparison["actual_AQI_nb6"],
            comparison["actual_AQI_nb7"],
            equal_nan=True,
            rtol=1e-6,
            atol=1e-8,
        ),
        "GAT-LSTM predicted AQI values match": np.allclose(
            comparison["predicted_AQI__gat_lstm"],
            comparison["predicted_AQI"],
            equal_nan=True,
            rtol=1e-6,
            atol=1e-8,
        ),
        "Actual AQI categories match": (
            comparison["y_true_aqi_nb6"].astype(str).to_numpy()
            == comparison["y_true_aqi_nb7"].astype(str).to_numpy()
        ).all(),
        "Predicted AQI categories match": (
            comparison["y_pred_aqi__gat_lstm"].astype(str).to_numpy()
            == comparison["y_pred_aqi"].astype(str).to_numpy()
        ).all(),
    }

    print("\nDetailed consistency checks:")
    for description, passed in checks.items():
        print(f"{'PASS' if passed else 'CHECK'}: {description}")

    if all(checks.values()):
        print(
            "\nSUCCESS: Notebook 7 uses the same AQI windows and "
            "matches Notebook 6's GAT-LSTM AQI results."
        )
    else:
        print(
            "\nThe station-time keys match, but some AQI values or "
            "categories differ. Check the AQI breakpoint definitions "
            "and post-processing in both notebooks."
        )


Notebook 6 windows: 84,014
Notebook 7 windows: 84,014
Identical station-time window sets: True

Detailed consistency checks:
PASS: Actual AQI values match
PASS: GAT-LSTM predicted AQI values match
PASS: Actual AQI categories match
PASS: Predicted AQI categories match

SUCCESS: Notebook 7 uses the same AQI windows and matches Notebook 6's GAT-LSTM AQI results.


In [17]:
# ============================================================
# CELL 11 — Final Validation
# ============================================================

assert N == 37
assert EDGE_INDEX.shape[1] == 169
assert INPUT_DIM == 46
assert SEQ_LEN == 24
assert len(FEATURES) == 34
assert len(TARGETS) == 6
assert nx.number_connected_components(G) == 1

print("="*60)
print("FINAL VALIDATION PASSED")
print("="*60)
print(f"Stations      : {N}")
print(f"Directed edges: {EDGE_INDEX.shape[1]}")
print(f"Features      : {len(FEATURES)}")
print(f"Input         : {INPUT_DIM}")
print(f"Targets       : {len(TARGETS)}")
print(f"Seq length    : {SEQ_LEN}")
print("Connected     : 1 component")
print("✓ Model artifacts saved")

FINAL VALIDATION PASSED
Stations      : 37
Directed edges: 169
Features      : 34
Input         : 46
Targets       : 6
Seq length    : 24
Connected     : 1 component
✓ Model artifacts saved


In [18]:

# ============================================================
# VERIFY SAVED CPCB BREAKPOINTS AND AVERAGING PERIODS
# ============================================================

from pathlib import Path
import joblib
import numpy as np

ROOT = Path.cwd()

while (
    not (ROOT / "models" / "gat_lstm_config.pkl").exists()
    and ROOT != ROOT.parent
):
    ROOT = ROOT.parent

CFG_PATH = ROOT / "models" / "gat_lstm_config.pkl"

if not CFG_PATH.exists():
    raise FileNotFoundError(
        f"Saved model configuration not found: {CFG_PATH}"
    )

cfg_check = joblib.load(CFG_PATH)
saved_bp = cfg_check["aqi_breakpoints"]

EXPECTED_WINDOWS = {
    "co": 8,
    "no2": 24,
    "o3": 8,
    "pm10": 24,
    "pm25": 24,
    "so2": 24,
}

EXPECTED_BP = {
    "pm10": [
        (0, 50, 0, 50),
        (50, 100, 50, 100),
        (100, 250, 100, 200),
        (250, 350, 200, 300),
        (350, 430, 300, 400),
        (430, np.inf, 400, 500),
    ],
    "pm25": [
        (0, 30, 0, 50),
        (30, 60, 50, 100),
        (60, 90, 100, 200),
        (90, 120, 200, 300),
        (120, 250, 300, 400),
        (250, np.inf, 400, 500),
    ],
    "no2": [
        (0, 40, 0, 50),
        (40, 80, 50, 100),
        (80, 180, 100, 200),
        (180, 280, 200, 300),
        (280, 400, 300, 400),
        (400, np.inf, 400, 500),
    ],
    "o3": [
        (0, 50, 0, 50),
        (50, 100, 50, 100),
        (100, 168, 100, 200),
        (168, 208, 200, 300),
        (208, 748, 300, 400),
        (748, np.inf, 400, 500),
    ],
    "co": [
        (0, 1, 0, 50),
        (1, 2, 50, 100),
        (2, 10, 100, 200),
        (10, 17, 200, 300),
        (17, 34, 300, 400),
        (34, np.inf, 400, 500),
    ],
    "so2": [
        (0, 40, 0, 50),
        (40, 80, 50, 100),
        (80, 380, 100, 200),
        (380, 800, 200, 300),
        (800, 1600, 300, 400),
        (1600, np.inf, 400, 500),
    ],
}

print("Configuration:", CFG_PATH)
print("\nSaved breakpoints:")

breakpoints_match = True

for pollutant, expected in EXPECTED_BP.items():
    actual = np.asarray(saved_bp[pollutant], dtype=float)
    expected_array = np.asarray(expected, dtype=float)

    matches = (
        actual.shape == expected_array.shape
        and np.allclose(actual, expected_array, equal_nan=True)
    )

    print(
        f"{pollutant.upper():5s}: "
        f"{'PASS' if matches else 'MISMATCH'}"
    )

    breakpoints_match &= matches

saved_windows = cfg_check.get("aqi_window_hours")

windows_match = (
    saved_windows is not None
    and saved_windows == EXPECTED_WINDOWS
)

print("\nSaved averaging periods:", saved_windows)
print(
    "Averaging-period check:",
    "PASS" if windows_match else "MISMATCH",
)


def calculate_subindex(values, pollutant, breakpoint_table):

    values = np.asarray(values, dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)

    valid = np.isfinite(values) & (values >= 0)

    for i, (lo, hi, ilo, ihi) in enumerate(
        breakpoint_table[pollutant]
    ):
        if i == 0:
            in_band = (values >= lo) & (values <= hi)
        elif np.isfinite(hi):
            in_band = (values > lo) & (values <= hi)
        else:
            in_band = values > lo

        use = valid & in_band & np.isnan(out)

        if np.isinf(hi):
            out[use] = 500.0
        else:
            out[use] = (
                ilo
                + (ihi - ilo) * (values[use] - lo) / (hi - lo)
            )

    return np.floor(out)


pm25_values = np.array([30, 31, 45, 60], dtype=float)

pm25_indices = calculate_subindex(
    pm25_values,
    "pm25",
    saved_bp,
)

print("\nCPCB worked-example check for PM2.5:")
for concentration, index in zip(pm25_values, pm25_indices):
    print(f"{concentration:5.1f} µg/m³ -> sub-index {index:.0f}")

example_match = np.array_equal(
    pm25_indices,
    np.array([50, 51, 75, 100], dtype=float),
)

print(
    "\nPM2.5 example check:",
    "PASS" if example_match else "MISMATCH",
)

if not (
    breakpoints_match
    and windows_match
    and example_match
):
    print(
        "\nWARNING: The saved configuration is inconsistent with "
        "the final AQI evaluation code. Do not use it for final "
        "visualization or live forecasting yet."
    )
else:
    print("\nSUCCESS: Saved AQI configuration verified.")


Configuration: c:\Users\vabhi\Downloads\Implementation\Implementation\models\gat_lstm_config.pkl

Saved breakpoints:
PM10 : PASS
PM25 : PASS
NO2  : PASS
O3   : PASS
CO   : PASS
SO2  : PASS

Saved averaging periods: {'co': 8, 'no2': 24, 'o3': 8, 'pm10': 24, 'pm25': 24, 'so2': 24}
Averaging-period check: PASS

CPCB worked-example check for PM2.5:
 30.0 µg/m³ -> sub-index 50
 31.0 µg/m³ -> sub-index 51
 45.0 µg/m³ -> sub-index 75
 60.0 µg/m³ -> sub-index 100

PM2.5 example check: PASS

SUCCESS: Saved AQI configuration verified.
